# 篇 4 · 内存的几本账与启动期就锁死的堆上限：哪几本在这台机器上**根本不存在**

> 要回答的坏结果：**"同一段 PySpark 代码在本地跑得好，进了 K8s 就 `OOMKilled` / 被限流，
> 到底哪几本账是我在这台机器上根本看不见的？"**
>
> 本篇不写"内存要配大一点"这种口号，而是把**五本账逐本读出来**，并且对每一本说清三件事：
> 它在这台机器上**存在吗**、用哪个 API 读它、以及读的时候**哪个坑会把"读不到"变成"读错"**。
> 最后那一条不是修辞：本篇实测到 `spark.conf.get(key, default)` 会**给你传的那个 default 做类型校验**
> （int 键上传字符串直接抛异常），也实测到它回的那个 default **顶替的是注册项自己的缺省值**
> —— 也就是说，"读不到就回个缺省"这条写了无数遍的 helper，在 SQLConf 上要么炸、要么骗人。
>
> ## 本篇的五本账（各一节，各一条结论 slug）
>
> | 节 | 那本账 | 在这台机器上 | slug |
> | --- | --- | --- | --- |
> | ① | driver 堆上限 | **存在，但只在启动期存在**（运行时改不动） | `heap-ceiling-is-boot-time` |
> | ② | 配置项的"生效值 vs 缺省值" | 存在，但双参 `get` 不是你以为的那条路 | `conf-default-is-type-checked` |
> | ③ | executor 的内存与核数 | **不存在**（local 模式没有 executor 这本账） | `local-mode-has-no-executor-account` |
> | ④ | Python 侧的峰值 | 存在，而且比结果本身大一个数量级 | `topandas-peaks-above-its-result` |
> | ⑤⑥ | container 的 cgroup 上限 | **本机没设** ⇒ `OOMKilled` 那一族不可观测 | `container-limit-is-absent-here` |
>
> ⚠ ④ 那条 slug 里那个 `P` 是**被迫小写**的：闸门的 slug 形状判据（`tutorials.test.ts` 常驻那组，
> 与它 `printedSlugs()` 的反向正则）只收 `[a-z0-9-]`，写作 `toPandas-…` 会**宿主档就地红**、
> 而且反向核对根本看不见那条 marker。改的是 slug 的大小写，不是判据。
>
> ## 本机边界（先说破，别拿本地的绿当 K8s 的证据）
>
> 这台机器上是 `local[2]` + `--driver-memory 512m` + **容器没有 memory limit**（三件都在 ①③⑥ 当场读出来，
> 不是照抄这段导语）。于是：
>
> - **driver 就是这个 kernel 进程**（local 模式下没有别的 JVM）。堆一炸整个 session 死，
>   后面的 cell 与闸门一起废 ⇒ **本篇只读边界，绝不撞边界**：没有任何"分配 600 MB 试试看会不会 OOM"的格子。
>   ④ 那格量到的峰值（约 0.6 GB 进程 RSS）是 `toPandas()` 转换过程的**自然结果**，两轮探针都跑通了，
>   不是刻意制造的事故。
> - **没有 Pod、没有 scheduler、没有 kubelet、没有 eviction、没有 `OOMKilled`**。
>   凡涉及那一层的账（`resources.limits.memory`、`memory.high` 与 OOM score、VPA 改配、
>   Spark on K8s 的 executor pod 内存、`local` 与 `cluster` 部署模式下 driver 内存归属的差别、
>   PySpark 的 python worker 进程是不是独立 pod）——**本篇一律标未验证**，不写成结论，
>   因为这台机器上没有任何一格能证明它们。它们归 WI-108 的另一半（`content/knowledge/` 语料）。
> - 三篇先例的分工照旧：**篇 1** 倾斜与热点 key、**篇 2** 小文件与物理设计、**篇 3** 计划解读与 AQE。
>   本篇**不复述**那三件（`shuffle.partitions` 怎么进计划、AQE 怎么合并、加盐怎么治倾斜都不在这里讲），
>   只讲内存与配置的账。
>
> 内核请选 `PySpark (arena)`（`metadata.kernelspec.name` = `arena-pyspark`）：`tutorials.test.ts` 容器档
> 会**真执行这一篇**并按内核名判，写成 `python3` 就跑到镜像自带的系统解释器上去了（红线①）。
> ⚠ 这个 kernel **既不预注入 `spark`、也不预导入 `SparkSession`**（本篇第一版探针就撞了 `NameError`），
> 所以下一格要自己建 session。最后一个单元是 `spark.stop()`：闸门跑完不留活内核 —— 这句话有判据
> （常驻那组那条结构性的，判的是"每篇最后一个 code cell 里有语句级 `spark.stop()`"，宿主档就能红）。

In [ ]:
import gc
import os
import resource
import sys
import time

from pyspark.sql import SparkSession, functions as f

spark = SparkSession.builder.appName("wi108-04-memory").getOrCreate()
sc = spark.sparkContext
rt = spark._jvm.java.lang.Runtime.getRuntime()


def proc_kb(name):
    '''/proc/self/status 里的某一行（单位 kB）。**self = 这个 kernel 进程 = local 模式下的 driver JVM**。

    为什么用它而不是 `resource.getrusage().ru_maxrss`：后者只有**高水位**一个数，量不到"当前"。
    ④ 那格要把两把尺并排放，正是因为选错尺会读错结论。
    '''
    with open("/proc/self/status", "r", encoding="utf-8") as fh:
        for ln in fh.read().splitlines():
            if ln.startswith(name + ":"):
                return int(ln.split(":", 1)[1].strip().split()[0])
    raise KeyError(name)


def heap_bytes():
    '''JVM 堆的"当前已用"= totalMemory() - freeMemory()。⚠ 它**不随 del/GC 回缩**（committed 的页留着复用），
    所以它是"用到过多少"的一个中间量，不是峰值 —— ④b 那格会当场把这一点打给读者看。'''
    return rt.totalMemory() - rt.freeMemory()


print("=" * 78)
print("[开局五行读数] 后面每一格都活在这几行之上（本镜像当场读的，不是背的）")
print("=" * 78)
print("spark version        ", spark.version)
print("master               ", sc.master)
print("defaultParallelism   ", sc.defaultParallelism, "   <- 活值；③ 那格判它与配置项的分叉")
print("Runtime.maxMemory()  ", rt.maxMemory(), "bytes =", rt.maxMemory() // (1024 * 1024), "MiB   <- ① 那格判它锁死")
print("availableProcessors  ", rt.availableProcessors(), "   <- 宿主机的核数，与 local[2] 无关（⑥ 只打印）")
print("sys.executable       ", sys.executable)
print("进程身份             ", "pid", os.getpid(), "ppid", os.getppid(), "（local 模式：这个 pid 就是 driver）")
print("PYSPARK_SUBMIT_ARGS  ", os.environ.get("PYSPARK_SUBMIT_ARGS", "(这个环境变量不在 kernel 的 os.environ 里)"))
print()
print("=> 五本账的地图（哪一本存在、哪一本不存在，见各节）")
print("   ① driver 堆上限          存在，但**只在启动期**存在（--driver-memory 来自上面那行 submit args）")
print("   ② 配置生效值 vs 缺省值    存在，但双参 get(key, default) 这条路不是你以为的那条")
print("   ③ executor 内存 / 核数    本机**不存在**（local 模式没有 executor 这本账）")
print("   ④ Python 侧的峰值         存在，且比结果本身大一个数量级（toPandas 的转换副本）")
print("   ⑤⑥ container cgroup 上限 本机**没设** => OOMKilled 那一族失效模式在这里不可观测")

## ① `heap-ceiling-is-boot-time`：driver 堆上限是**启动期**锁死的实体，不是运行时可改的配置

一句话：**`--driver-memory` 是 JVM 的启动参数，不是 Spark 的配置项。** JVM 一旦起来，
`-Xmx` 就焊死在那个进程上了；Spark 侧的 `spark.driver.memory` 只是那次启动的**记录**，
不是一个可以被"重新应用"的旋钮。

为什么这件事在 local 模式尤其致命：**driver JVM 就是你现在按 shift+enter 的这个进程**。
在 `spark-submit --deploy-mode cluster` 或 K8s 上，driver 崩了还有另一个进程（driver pod）
可以重启、可以留日志；这里堆炸了，kernel 直接没了，页面上表现为"这个 notebook 断了"，
而**没有任何 OOM 报告可看**（JVM 的 `OutOfMemoryError` 是 Java 异常，不是内核的 SIGKILL，
这两者的区别正是 ⑥ 那一节的分界）。

所以 ① 读的是**边界**，不是**事故**。下面那格刻意不做任何"分配 600 MB 看看会不会炸"的实验 ——
那是本篇的纪律：**只读边界，不撞边界**（撞一次的代价是整场 session 与后面所有格子）。

三条判据都是确定量：`maxMemory()` 逐字节等于 `512*1024*1024`（顺带量了"512m 这个字符串被解析成 512 MiB"）、
运行时 `set` 它**抛** `AnalysisException` 且消息含 `CANNOT_MODIFY_CONFIG`、抛过之后 `maxMemory()` 一字不变。
⚠ 第二条不许写成裸 `except: pass`：吞掉之后"没抛"与"抛了但没人判"在屏幕上一模一样，
所以这里接住异常，然后**逐条判它的类型和消息**。

In [ ]:
# marker heap-ceiling-is-boot-time gate 的是**一句**结论："driver 堆上限是启动期锁死的实体，
# 不是运行时可改的配置"。下面五条 assert 是这句话的五面（缺任何一面，那句话都不成立）：
#   (a) 上限逐字节 = 512 MiB          —— 那句 --driver-memory 512m 真的落成了堆上限
#   (b) 运行时改它**抛**              —— 不是"改了不生效"，是 API 层面明确拒绝
#   (c) 拒绝的异常类型是 AnalysisException —— 拒绝的原因有一百种，只判"抛了"会替一件错事作证
#   (d) 消息含 CANNOT_MODIFY_CONFIG   —— 钉的正是"这个键不可改"，不是碰巧的别的错
#   (e) 抛过之后 maxMemory() 不变      —— "没生效"要有读数，不能只靠正文那句断言
before_bytes = rt.maxMemory()
conf_at_boot = spark.conf.get("spark.driver.memory")
print("maxMemory() 之前        =", before_bytes, "bytes =", before_bytes / (1024 * 1024), "MiB")
print("已用堆 heap_bytes()     =", heap_bytes(), "bytes（对比用：它是'当前已用'，不是上限）")
print("spark.driver.memory     =", repr(conf_at_boot), "<- 启动期那个值的**记录**")

set_exc = None
try:
    spark.conf.set("spark.driver.memory", "2g")
except Exception as e:  # 接住"这一次 set"，下面 (c)(d) 逐条判它；不许裸 except 吞掉
    set_exc = e
after_bytes = rt.maxMemory()
print("试着 set('spark.driver.memory', '2g') =>", "没抛（！与正文不符）" if set_exc is None else type(set_exc).__name__)
if set_exc is not None:
    print("   消息第一行:", str(set_exc).splitlines()[0])
print("maxMemory() 之后        =", after_bytes, "bytes | 变了没有:", after_bytes != before_bytes)
print("SparkConf 侧读回来的     =", repr(sc.getConf().get("spark.driver.memory", "(unset)")), "（还是启动期那个）")

# 一条**纯算式**（明确不是实测容量，别当结论背）：把堆上限除以 8 字节，得到"裸 bigint 能放多少个"的上界。
print()
print("算式对照（**不是实测**）：上限 / 8 B =", before_bytes // 8, "个 bigint。")
print("   真实的每一行要贵一个量级 —— ④b 那格会用它自己量到的 bytes/行 把这个数算一遍给读者看。")

assert before_bytes == 512 * 1024 * 1024, (
    f"maxMemory() = {before_bytes} != 512 MiB ⇒ kernel.json 里那句 --driver-memory 512m 变了（镜像被换过）。"
    "本篇后面每格都拿这个数当分母（④ 的峰值账、导语那句'driver 就是这个 kernel 进程'），"
    "换了就要整篇重测 —— 改的是镜像与本篇的说明，别把这条改成'读到多少算多少'"
)
assert set_exc is not None, (
    "spark.conf.set('spark.driver.memory', '2g') 居然没抛 ⇒ 它变成了一个运行时可改的键。"
    "本节正文那句'启动期锁死的实体'当场过期，而且堆上限可能真的跟着动了：先看上面 (e) 那条读数"
)
assert type(set_exc).__name__ == "AnalysisException", (
    f"拒绝的类型是 {type(set_exc).__name__}，不是 AnalysisException ⇒ 版本行为漂了，"
    "正文那句要按这里读到的类型改写（别删这条 assert 让它闭嘴）"
)
assert "CANNOT_MODIFY_CONFIG" in str(set_exc), (
    f"抛了但消息里没有 CANNOT_MODIFY_CONFIG（现在是 {str(set_exc).splitlines()[0]!r}）"
    "⇒ 挡住你的是**另一件**坏事（比如键名拼错、权限），本节那句结论没被这条判据证明过"
)
assert after_bytes == before_bytes, (
    f"set 被拒之后 maxMemory() 从 {before_bytes} 变成 {after_bytes} ⇒ 出现'抛了但仍然改了'的形状，"
    "那比'锁死'糟得多（运行时静默改堆上限 = 每次跑的峰值账都对不上），本篇正文要整段重写"
)
print("WI90[04-memory-accounts-and-the-boot-time-ceiling][heap-ceiling-is-boot-time] OK", flush=True)

## ② `conf-default-is-type-checked`：双参 `get(key, default)` 会给**默认值**做类型校验，
## 而它回的那个 default **顶替的是注册项自己的缺省值**

这是本篇操作性最强的一节，因为坏法藏在最普通的写法里：

```python
partitions = spark.conf.get("spark.sql.shuffle.partitions", "200")   # 看起来人畜无害
```

它比大家以为的两件事都离谱：

1. **默认值要过这个键的类型转换。** int 型键上传一个非整数的 default，抛
   `IllegalArgumentException: spark.sql.shuffle.partitions should be int, but was (unset)` —— 不是回默认值。
   boolean 型键同形（`should be boolean`）。字符串型键**不过**校验，于是"回默认"这条路在这里安静地通。
2. **即便它不抛，回给你的也不是"当前生效值"。** 双参形式只在**raw 层**找值；raw 层没有就直接用你传的
   default，**注册项自己的缺省值在这条路上一个字都不参与**。所以
   `get(k)` 给 `200`（注册项缺省）而 `get(k, "999")` 给 `999` —— 同一个键、同一个瞬间、两个答案，
   后者还**不抛**。`get("spark.sql.adaptive.enabled", "false")` 回 `"false"` 而 AQE 实际是 `"true"`，
   这就是"读配置来判断行为"这类 helper 最坏的一种坏法：**它给一个自洽的错答案**。

机制上的分界线是"raw 层有没有值"，所以本篇把那三条腿都跑了一遍（②-⑧ 那行）：一旦显式
`set` 过，双参就回到"读 raw"，同一个 `"(unset)"` 默认值这次**不炸**了 ⇒ 抛的原因是"默认值要过类型转换"，
不是"这个键读不出来"。⚠ 这一条不是从文档推的，是下面那格现场做 set / unset 量出来的。

**为什么这属于"内存账"而不是一节 API Trivia**：`spark.driver.memory` / `spark.executor.memory` /
`spark.default.parallelism` 这几个键在 SQLConf 与 SparkConf 里的**存在性不一样**（③ 那节就是它），
而一个人排查内存问题时写的第一行 helper 通常是"读不到就算个缺省" —— 那条 helper 在这三个键上
要么抛、要么给你一个和生效值无关的串。

⚠ 与篇 3 的分工：篇 3 的开局那格在**注释**里顺手记过"给整型键传字符串 default 直接抛"这一句
（它当时的目的是解释"为什么那里不传 default 而是接异常"）。本篇把同一件事**做成有 slug 的结论**，
并补上篇 3 没量的三格对照（合法整数 default 会顶替注册项缺省 / boolean 键同形 / raw 有值时 default 被忽略）。
不是复述篇 3 的结论，是把那句话变成有判据的对象。

In [ ]:
# marker conf-default-is-type-checked gate 的是**一句**结论（② 那节标题那两行）：
#   双参 get(key, default) 会给 default 做类型校验，且它回的 default 顶替注册项自己的缺省值。
# 下面这些 assert 全是这一句的不同侧面，按"键的类型 × raw 层有没有值"摆成一张表：
#   int 键  raw 无  非整数 default -> 抛 IllegalArgumentException（正面：default 要过转换）
#   int 键  raw 无  整数   default -> 回你传的，**不是**注册项缺省（正面：default 顶替注册项）
#   bool 键 raw 无  文本   default -> 抛 should be boolean；bool 键 raw 无 'false' -> 回 false（骗人的那一格）
#   字符串键 raw 有 (driver.memory) -> 回 raw 的 512m，你的 default 被忽略（简报点名的那格对照）
#   字符串键 raw 无 (sql.extensions)-> 单参回 Python None、双参回 '(unset)'（"回默认"在这里是骗人的）
#   非 SQL 键 (executor.memory)     -> 一个校验都不走，直接回 default（交给 ③ 判它的存在性）
#   set 之后再双参 -> 回 set 的值（把"炸的原因是默认值过转换"钉死，并顺手恢复出厂）
K_INT = "spark.sql.shuffle.partitions"          # 注册项，int 型；出厂没有 raw 值 => 生效值 200 来自注册项缺省
K_BOOL = "spark.sql.adaptive.enabled"           # 注册项，boolean 型；同上
K_STR_RAW = "spark.driver.memory"               # raw 层有值（启动期从 submit args 进来）
K_STR_NONE = "spark.sql.extensions"             # 注册项，字符串型，从没设过
K_NOT_SQL = "spark.executor.memory"             # **不是** SQL 配置（存在性问题交给 ③）

one_int = spark.conf.get(K_INT)
two_int_ok = spark.conf.get(K_INT, "999")
one_bool = spark.conf.get(K_BOOL)
two_bool_false = spark.conf.get(K_BOOL, "false")
two_bool_verbatim = spark.conf.get(K_BOOL, "FALSE")
two_str_raw = spark.conf.get(K_STR_RAW, "(unset)")
one_str_none = spark.conf.get(K_STR_NONE)
two_str_none = spark.conf.get(K_STR_NONE, "(unset)")
two_not_sql = spark.conf.get(K_NOT_SQL, "2g")

int_exc = None
try:
    spark.conf.get(K_INT, "(unset)")
except Exception as e:
    int_exc = e
bool_exc = None
try:
    spark.conf.get(K_BOOL, "(unset)")
except Exception as e:
    bool_exc = e

print("int 键   raw 无  单参          ->", repr(one_int), "（注册项缺省）")
print("int 键   raw 无  双参 '999'    ->", repr(two_int_ok), "<- 不是上面那个 200：你传的顶替了注册项缺省")
print("int 键   raw 无  双参 '(unset)'->", "抛 " + type(int_exc).__name__ + ": " + str(int_exc).splitlines()[0] if int_exc else "没抛（与正文不符）")
print("bool 键  raw 无  单参          ->", repr(one_bool))
print("bool 键  raw 无  双参 '(unset)'->", "抛 " + type(bool_exc).__name__ + ": " + str(bool_exc).splitlines()[0] if bool_exc else "没抛（与正文不符）")
print("bool 键  raw 无  双参 'false'  ->", repr(two_bool_false), "<- 同一个键同一个瞬间两个答案，而且**不抛**")
print("bool 键  raw 无  双参 'FALSE'  ->", repr(two_bool_verbatim), "（回的是你传的那个串，逐字，不做归一化）")
print("字符串   raw 有  双参 '(unset)'->", repr(two_str_raw), "<- raw 赢了，你的 default 被忽略（简报点名的对照）")
print("字符串   raw 无  单参          ->", repr(one_str_none), "(Python None；SET -v 里那一行是 <undefined>)")
print("字符串   raw 无  双参 '(unset)'->", repr(two_str_none))
print("非 SQL   键      双参 '2g'     ->", repr(two_not_sql), "（一个类型校验都没走；它压根不是 SQLConf 的键，见 ③）")

# 机制的分界线：raw 层一旦被显式 set，双参就回到"读 raw"这条路 —— 同一个 '(unset)' 默认值这次不炸。
# 这一段自己收尾（set 了就 unset），不留配置残留给后面的格子。
spark.conf.set(K_INT, 12)
two_int_after_set = spark.conf.get(K_INT, "(unset)")
spark._jsparkSession.conf().unset(K_INT)
one_int_restored = spark.conf.get(K_INT)
print("显式 set 成 12 后 双参 '(unset)'->", repr(two_int_after_set), "<- 炸的不是这个键，是'默认值过类型转换'")
print("unset 之后       单参          ->", repr(one_int_restored), "（本篇已把 " + K_INT + " 恢复出厂）")

assert one_int == "200", f"{K_INT} 单参读到 {one_int!r}，不是 200 ⇒ 这台镜像的注册项缺省变了，上面那张表整张要重读"
assert int_exc is not None and type(int_exc).__name__ == "IllegalArgumentException", (
    f"int 键上传非整数 default 的结果不是 IllegalArgumentException（现在是 {type(int_exc).__name__ if int_exc else '没抛'}）"
    "⇒ 那句'读不到就回缺省'的坏法可能已经被修好了，本篇 ② 的正文要按这里读到的改写"
)
assert "should be int" in str(int_exc), f"抛了但消息里没有 'should be int'（现在是 {str(int_exc).splitlines()[0]!r}）⇒ 是另一件坏事"
assert two_int_ok == "999", (
    f"双参传合法整数 default 回的不是 '999' 而是 {two_int_ok!r} ⇒ 双参形式开始尊重注册项缺省了，"
    "本节'它顶替的是注册项自己的缺省值'那半句过期（这条 assert 是本节最容易随小版本翻脸的一条，"
    "红了先改正文，别改期望值）"
)
assert one_bool == "true" and two_bool_false == "false", (
    f"bool 键那一格的分叉不见了（单参 {one_bool!r} / 双参 'false' 回 {two_bool_false!r}）"
    "⇒ '给一个自洽的错答案'这条最坏的坏法在这台镜像上不成立，正文要改"
)
assert bool_exc is not None and "should be boolean" in str(bool_exc), f"boolean 键不同形了（现在是 {str(bool_exc).splitlines()[0] if bool_exc else '没抛'!r}）⇒ 类型校验只挂在 int 上？正文那句要重写"
assert two_bool_verbatim == "FALSE", f"双参把 default 归一化了（回 {two_bool_verbatim!r}）⇒ '逐字回你传的那个串'那句过期"
assert two_str_raw == "512m", (
    f"字符串键且 raw 有值时双参没有忽略 default（回 {two_str_raw!r}）⇒ raw 与 default 的优先级变了，"
    "上面那张表与 ① 那节的'set 被拒'两处的解释都要重读"
)
assert one_str_none is None and two_str_none == "(unset)", (
    f"从没设过的字符串键的两种读法变了（单参 {one_str_none!r} / 双参 {two_str_none!r}）"
    "⇒ '回默认在这里是骗人的'那一句要按新读数改写"
)
assert two_not_sql == "2g", f"非 SQL 键的双参不再直接回 default（{two_not_sql!r}）⇒ 它被注册进 SQLConf 了，③ 那节的存在性表要重做"
assert two_int_after_set == "12", (
    f"显式 set 之后双参仍回 default（{two_int_after_set!r}）⇒ 双参**永远**忽略当前值，"
    "那是比本节正文说的更狠的形状，正文与 ② 那张表要按这一行重写"
)
assert one_int_restored == "200", (
    f"unset 之后没回到注册项缺省（{one_int_restored!r}）⇒ 本篇给后面的格子留了配置残留，"
    "红在这里说的是上面那行 unset 没生效，不是'200 这个数字不重要'"
)
print("WI90[04-memory-accounts-and-the-boot-time-ceiling][conf-default-is-type-checked] OK", flush=True)

## ③ `local-mode-has-no-executor-account`：local 模式下"executor 内存"这本账**压根不存在**

排查内存问题的人第一行代码通常是"看看 executor 配了多少"。在 `local[*]` 系列上，
**这个问题没有答案**，而"没有答案"有两种完全不同的形状，读的人必须分辨：

| 读法 | 结果 | 意思 |
| --- | --- | --- |
| `spark.conf.get("spark.executor.memory")` | **抛** `SparkNoSuchElementException: [SQL_CONF_NOT_FOUND]` | SQLConf 里**没有这个键** |
| `sc.getConf().get("spark.executor.memory", "(unset)")` | 回 `"(unset)"` | SparkConf 里也**没有这个值**（不是"读法不对"） |
| `sc.getConf().get("spark.driver.memory")` | 回 `"512m"` | driver 那本账**存在且有名字** |
| `sc.defaultParallelism` | 回 `2`（活值） | 并行度**在用**，但 `spark.default.parallelism` 这个配置项**从没设过** |

最后一行是本节真正要教的形状：**"有活值"与"有配置项"是两件事**。
`local[2]` 的那个 `2` 来自 master 字符串，`sc.defaultParallelism` 直接把它作为整数交给你，
而 `spark.default.parallelism` 在 SparkConf 里查无此键。于是两种误判都会发生：

- 有人用 `get(..., 缺省)` 去"读并行度"，读到缺省值就以为"没配 ⇒ 用默认 8"，
  而真实并行度是 2（再叠上 ② 那一节：双参 `get` 在 int 键上还会直接抛）。
- 有人把"读不到 executor 内存"读成"这台机器内存没配/配错了"，
  而正确的说法是 **local 模式没有 executor 这本账**（`local[2]` = 一个 JVM 里的 2 个线程）。

⇒ 迁移到 K8s / YARN 时，这三本账（executor 内存、executor 核数、`default.parallelism`）
会从"不存在"变成"存在而且会被独立记账"。**这个"从不存在到存在"的差别，本机演示不了（未验证）** ——
本篇能证的只有"此刻这台机器上它们不存在"，而那恰恰是标题那句坏结果的另一半：
你在本地**看不见**的那几本账，不是"还没配好"，是**根本没有**。

In [ ]:
# marker local-mode-has-no-executor-account gate 的是**一句**结论：
#   local 模式下"executor 内存"这本账压根不存在，而 defaultParallelism 有活值却没有配置项。
# 五条 assert 各面：活值在（并行度确实是 2）/ 同一个键换个读法就没了（SparkConf 查无此项）/
#   SQLConf 里连键都没有（抛，而且是 SQL_CONF_NOT_FOUND 而不是"值为空"）/
#   SparkConf 侧也证实没有（排除"只是读法不对"）/ driver 那本账存在且有名字（对照组）。
print("master                    =", sc.master, "<- local[2] 的那个 2 就是并行度的来源")
live_parallelism = sc.defaultParallelism
conf_parallelism = sc.getConf().get("spark.default.parallelism", "(unset)")
print("sc.defaultParallelism     =", live_parallelism, "(活值)")
print("sc.getConf().get(该键)     =", repr(conf_parallelism), "(同一个东西，两处读法不一样)")

executor_exc = None
try:
    spark.conf.get("spark.executor.memory")
except Exception as e:
    executor_exc = e
msg = "" if executor_exc is None else str(executor_exc)
found_line = next((ln.strip() for ln in msg.splitlines() if "SQL_CONF_NOT_FOUND" in ln), "(消息里没有 SQL_CONF_NOT_FOUND 这一行)")
print("spark.conf.get(executor.memory) =>", "没抛（与正文不符）" if executor_exc is None else type(executor_exc).__name__)
print("   那一条:", found_line)

sc_executor_memory = sc.getConf().get("spark.executor.memory", "(unset)")
sc_executor_cores = sc.getConf().get("spark.executor.cores", "(unset)")
sc_driver_memory = sc.getConf().get("spark.driver.memory", "(unset)")
print("sc.getConf().get(executor.memory, '(unset)') =", repr(sc_executor_memory))
print("sc.getConf().get(executor.cores,   '(unset)') =", repr(sc_executor_cores), "(只打印：与 memory 同形，本节不为它单独发判据)")
print("sc.getConf().get(driver.memory)               =", repr(sc_driver_memory), "<- 对照：driver 这本账存在且有名字")
print()
print("=> 本机存在的内存账只有一本（driver 的 " + str(rt.maxMemory() // (1024 * 1024)) + " MiB）+ Python 侧那本（④）。")
print("   K8s 上的 executor pod 内存 / limit / request 那几本，这台机器上**没有对应的读法**（未验证，见导语边界）。")

assert live_parallelism == 2, (
    f"defaultParallelism = {live_parallelism} ⇒ kernel 不再是 local[2]（镜像或 kernel.json 被换过）。"
    "本篇'并行度'意义上的每一句话都以 2 为分母，换机器要整篇重测，别改这条断言去迁就新数字"
)
assert conf_parallelism == "(unset)", (
    f"spark.default.parallelism 现在有值了（{conf_parallelism!r}）⇒ '有活值却没有配置项'这条分叉没了，"
    "本节那张表的最后一行与'两种误判'的第一条都要按这里改写"
)
assert executor_exc is not None, (
    "spark.conf.get('spark.executor.memory') 不再抛 ⇒ 这台镜像把 executor 的账注册进 SQLConf 了，"
    "本节标题那句'这本账压根不存在'当场过期（正文要重写，别删这条 assert）"
)
assert "SQL_CONF_NOT_FOUND" in msg, (
    f"抛了，但原因不是'键不存在'（{found_line!r}）⇒ 那是另一件坏事，本节拿它当'没有这本账'的证据是不成立的"
)
assert sc_executor_memory == "(unset)", (
    f"SparkConf 侧现在有 executor.memory 了（{sc_executor_memory!r}）⇒ 不是'读法不对'而是'这本账开始存在'，"
    "本节那张表的第二行要改"
)
assert sc_driver_memory == "512m", (
    f"sc.getConf().get('spark.driver.memory') = {sc_driver_memory!r}，不是 '512m' ⇒ 对照组失效，"
    "① 那节'配置记录与 JVM 上限同源'的解释要重读（那里判的是 maxMemory()==512 MiB，这里判的是记录）"
)
print("WI90[04-memory-accounts-and-the-boot-time-ceiling][local-mode-has-no-executor-account] OK", flush=True)

## ④ `topandas-peaks-above-its-result`：`toPandas()` 花的是**转换过程中的副本**，不是结果的大小

`toPandas()` 的常见误解是"它的内存账 = 那个 DataFrame 的大小"。错两处：

1. **峰值由转换过程决定，不由结果决定。** 本镜像没有 pyarrow（⑤ 那节），
   `toPandas()` 走的是 `collect()` 那条老路：数据先在 **JVM 侧**被物化成 Java 对象、
   再逐行过 py4j 的线，最后才拼成 pandas 的列。所以峰值是"结果 + 途中那些副本"，
   而结果只是其中很小的一份。④a 那格把它量成比数：**峰值增量是净结果的 35~36 倍**（六次实测的范围），
   判据只押一个保守下界 `>= 4 倍` —— 因为**倍数本身会随版本与形状漂**（带 pyarrow 的那条路就不同），
   而下界才是"副本远大于结果"这句话的内容。
2. **它的账不只记在 JVM 堆里。** ④b 那格把 `collect()` 与 `toPandas()` 各自的 heap 增量与 RSS 增量并排量
   给读者看，并且**先把"基线本身可信"这件事量出来**：同一格会打印 `settle`（Python `gc.collect()` +
   JVM `System.gc()`）前后的"已用堆"之差 —— 那就是挂在读数上的垃圾。第一轮探针在**干净 kernel**
   （没有 ④a 那一笔在前）上量到的分叉是：`collect()` 堆 +75,360,096 B / RSS +55,352 kB，
   而 `toPandas()` 堆只 +18,268,656 B、RSS 却 +49,504 kB ⇒ **堆增量小得多而 RSS 差不多**，
   因为 pandas 那一列的账记在另一半（numpy 的缓冲区 + 过 py4j 线时的对象）。
   ⇒ 只看 `spark.driver.memory` 或只看 JVM 的堆读数，会得出片面的结论。
   ⚠ 这一组数**只打印、不 assert**，而且方向都别押：这一格最初版没 settle 基线，`collect()` 的
   "堆增量"量出来是 **-1,820,560 B（负数）** —— 那不是省内存，是上一格的 200 MB 垃圾被这一次 GC
   顺手收走了。**"增量"必须先问"基线是什么时候读的"**，这是本节最实在的一课，所以我把那次坏读数
   写在 ④b 那格的头一段注释里（它是历史读数，本轮打印的是 settle 前后的差），而不是假装没发生过。
   ④a 自己那格的堆增量（2,000,000 行那一次约 +200 MB）比 300,000 行的 `collect()` 还大 ——
   两件事不矛盾：**堆与 Python 两本账都存在**，谁都不等于"结果的大小"。

顺带一条仪器课，它是本节最容易读错数的地方，而**能押的与不能押的要分清**：

- **高水位永不掉**：`del pdf; gc.collect()` 之后 `/proc/self/status` 的 `VmHWM` 不减（它是单调量，
  这是内核定义，不是运气），`resource.getrusage().ru_maxrss` 与它是同一个数。
  ⇒ "我峰值到过哪里"必须用 HWM 证明。
- **同一瞬间 `VmHWM` 与 `VmRSS` 差一个数量级**（六次实测 457,000~461,000 kB），这才是可押的那一条：
  它说的就是"当前的量"与"到过的量"是两个东西。
- ⚠ **`del` 之后 `VmRSS` 掉多少，本篇不押**：六次实测分别是 +3,020 / +1,696 / +1,712 / +732 / +700 /
  **-324** kB —— 最后一次不降反升。原因是 16 MB 的释放在 OS 层面只有 MB 级效果（glibc 不一定把页
  还给内核），而 Spark 后台线程在同一窗口里碰的页就能把它盖掉。这一条曾在判据里，容器档红过一次，
  于是被换成上面那条"同一瞬间的差"。**教训比结论值钱**：想证明"我释放了"，去看 RSS 的**量级**与
  `VmHWM - VmRSS` 那个差，别去押 MB 级的方向。

⚠ **两条纪律在本节的落点**：
- **不 assert 耗时**（④a 会打印 `toPandas()` 的秒数，但一条都不押）。Ruling(B1) 在三篇先例里否掉了
  押时间的断言，本篇照办：墙上时间在这台机器上受 JIT 预热、别的 notebook 残留、宿主调度影响，
  押它只会造出与结论无关的冤红。
- **不制造 OOM**。④a 用的是 N = 2,000,000 的单列 `int64`（16 MB 净结果），
  这个形状在 WI-108 的两轮探针与本篇落地时的六次独立运行里都跑通了（进程峰值 RSS 637~645 MB，
  而 `memory.max` 是 `max` ⇒ 没有可撞的上限），本篇只是复现同一个形状，不去撞堆上限。
  ⚠ 顺序是刻意的：**重的 ④a 排在 ④b 之前**，让 kernel 处在与探针**相同**的状态
  （前面只有配置读写，没有大对象），这样"跑通"这件事的依据是那一轮实测而不是我的推测。
- **净大小可以纯算**（④a 第一条 assert）：`memory_usage(deep=True).sum()` = 行数 × `dtype.itemsize`
  **加上 index 那一份**。那"加一份"是量出来的（`RangeIndex` 记 128 字节），不是推测 ——
  ⚠ 简报 §3 那条判据写的是 `== N * 8`，实测 `16,000,128 != 16,000,000`，差的正是 index 那 128 B；
  所以等号按读数改成"可纯算的那一条"（见报告"简报里哪一句是错的"）。

In [ ]:
# marker topandas-peaks-above-its-result gate 的是**一句**结论：toPandas() 花的是转换过程中的副本，
# 不是结果的大小。七条 assert 各面（全是确定量、可纯算式或定义保证的量；**一条都不押耗时**）：
#   (1) 行数对得上          —— 不是 2M 行就没有这本账可算（前提）
#   (2) 列的 deep 用量 = N * itemsize —— "净大小"里那一列是纯算的
#   (3) 总和 = 列 + index     —— 净大小可纯算（实测 index 记 128 B；简报那条 == N*8 的等号差的就是它）
#   (4) 峰值增量 >= 4 * 净    —— 保守下界（六次实测 35~36 倍；押 4 是因为**倍数本身随版本漂**，
#                              而"副本远大于结果"这句话的内容是下界，不是那个倍数）
#   (5) |ru_maxrss - VmHWM| <= 2MB —— 两把尺量的是同一个高水位（仪器一致性；容差的理由见下面那条 assert）
#   (6) hwm_did_not_drop is True —— del + gc 之后高水位不减（定义保证，不是运气）
#   (7) 同一瞬间 VmHWM - VmRSS >= 100MB —— "两把尺差一个数量级"才是可押的那一条
# ⚠ 曾经押过、被实测撤掉的一条：`assert rss_rel < rss_after`（"del 之后 VmRSS 会掉"）。
#   六次实测的降幅是 +3,020 / +1,696 / +1,712 / +732 / +700 / **-324** kB —— 最后一次不降反升，
#   于是它在闸门里红过一次（容器档 cell 9 AssertionError）。MB 级的释放会被后台线程碰页盖掉，
#   押它得到的不是结论，是一条每次跑都可能翻转的红 ⇒ 改成 (7) 那种"同一个瞬间的差"，只打印降幅。
N = 2_000_000
t_boot_check = time.time()
df = spark.range(N, numPartitions=8).toDF("id")
df.cache().count()  # 先落进缓存：让 toPandas 那次转换读的是同一份数据（与探针同一形状）
print("造数据 + cache + count:", round(time.time() - t_boot_check, 2), "s（**只打印**）")

hwm_before = proc_kb("VmHWM")
rss_before = proc_kb("VmRSS")
heap_before = heap_bytes()

t0 = time.time()
pdf = df.toPandas()
secs_to_pandas = round(time.time() - t0, 2)
hwm_after = proc_kb("VmHWM")
rss_after = proc_kb("VmRSS")
ru_after = int(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss)

rows = int(len(pdf))
itemsize = int(pdf["id"].dtype.itemsize)
index_bytes = int(pdf.index.memory_usage())
usage = pdf.memory_usage(deep=True)
col_bytes = int(usage["id"])
deep_bytes = int(usage.sum())
net_kb = deep_bytes / 1024
peak_kb = hwm_after - hwm_before
ratio = peak_kb / net_kb

print("toPandas() 墙上时间   =", secs_to_pandas, "s（只打印，不进任何判据）")
print("dtype / itemsize      =", str(pdf["id"].dtype), "/", itemsize, "B")
print("rows                  =", rows)
print("列的 deep 用量         =", col_bytes, "B  =  N * itemsize =", N * itemsize, "B")
print("index（" + type(pdf.index).__name__ + ") 的 deep 用量 =", index_bytes, "B   <- 净大小里**别忘了它**")
print("memory_usage(deep).sum =", deep_bytes, "B  （所以 '== N*8' 这条等号是错的，差的就是上面那份）")
print("VmHWM  前 / 后         =", hwm_before, "/", hwm_after, "kB")
print("VmRSS  前 / 后         =", rss_before, "/", rss_after, "kB")
print("ru_maxrss（toPandas 后）=", ru_after, "kB   <- 与上面那个 VmHWM 是同一个数（差的绝对值 =", abs(ru_after - hwm_after), "kB）")
print("峰值增量 / 净大小       =", round(peak_kb, 1), "kB /", round(net_kb, 1), "kB =", round(ratio, 1), "倍")
print("堆增量 heap_bytes()    =", heap_bytes() - heap_before, "B（**基线没 settle**，所以它只是量级参考；")
print("                        为什么必须 settle 见 ④b —— 那一格会把'挂在已用堆上的垃圾'当场量出来）")

del pdf
gc.collect()
hwm_rel = proc_kb("VmHWM")
rss_rel = proc_kb("VmRSS")
ru_rel = int(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss)
df.unpersist()
hwm_did_not_drop = hwm_rel >= hwm_after
print()
print("del pdf + gc.collect() 之后：")
print("  VmHWM      =", hwm_rel, "kB  (变化", hwm_after - hwm_rel, "kB) <- 高水位**不掉**（它是单调的，这是定义）")
print("  VmRSS      =", rss_rel, "kB  (变化", rss_after - rss_rel, "kB) <- '我释放了'要看这个数，但它的**方向不保证**")
print("  ru_maxrss  =", ru_rel, "kB   (还是那个峰值)")
print("  同一瞬间 VmHWM - VmRSS =", hwm_rel - rss_rel, "kB  <- 同一个进程、同一个时刻，两把尺差一个数量级")
print("  ⚠ 上面那行 'VmRSS 变化' 本篇**不 assert**：六次实测分别是 +3,020 / +1,696 / +1,712 / +732 / +700 / -324 kB，")
print("     最后一次是**涨了** 324 kB —— 释放在 OS 层面只有 MB 级、还被后台线程碰页盖掉，押它就是一条冤红。")

assert rows == N, f"结果是 {rows} 行，不是 {N} ⇒ 净大小与峰值都对着不上对象，下面那条比值没有意义（先查 spark.range 那行）"
assert col_bytes == N * itemsize, f"列的 deep 用量 {col_bytes} != N * itemsize = {N * itemsize} ⇒ pandas 的记账方式变了，本节的'净大小可纯算'要重写"
assert deep_bytes == col_bytes + index_bytes, (
    f"memory_usage(deep=True).sum() = {deep_bytes} != 列 {col_bytes} + index {index_bytes}"
    " ⇒ 简报那句 '== N * 8' 的形状与 pandas 现在的算法都还没被这条判据说清，正文要按新读数改写"
)
assert peak_kb >= 4 * net_kb, (
    f"峰值增量 {peak_kb:.0f} kB 只有净大小 {net_kb:.0f} kB 的 {ratio:.1f} 倍，低于 4 倍下界"
    " ⇒ 转换途中的副本被优化掉了（带 pyarrow 的那条路很可能就是这个形状），"
    "本节标题那句'花的不是结果的大小'要按新倍数改写 —— 红了先改正文，别把下界调小"
)
assert abs(ru_after - hwm_after) <= 2048, (
    f"ru_maxrss {ru_after} 与 VmHWM {hwm_after} 差过 2 MB ⇒ 两把尺不再同源（上面那行打印的是确切差值），"
    "上面那段'选错尺'的讲法要按这里重写。⚠ 容差 2 MB 不是和稀泥：两次读数之间后台线程碰过页就会差几十 kB，"
    "押逐字相等是另一种冤红（本篇六次实测的差值都是 0）"
)
assert hwm_did_not_drop is True, (
    f"VmHWM 竟然掉了（{hwm_after} -> {hwm_rel}）⇒ 内核语义/读法变了，本节'高水位不掉'那段依据要重读"
)
assert hwm_rel - rss_rel >= 100_000, (
    f"同一瞬间 VmHWM - VmRSS 只有 {hwm_rel - rss_rel} kB，低于 100,000 kB 的下界"
    " ⇒ '当前用量'已经追上了'峰值'，本节那句'两把尺差一个数量级'过期（六次实测这个差是 457,000~461,000 kB，"
    "所以下界押 1/4 就够；红了先改正文，别把下界调到贴着读数）"
)
print("WI90[04-memory-accounts-and-the-boot-time-ceiling][topandas-peaks-above-its-result] OK", flush=True)

In [ ]:
# ④b **只打印、不 assert**（也不发 marker）：把 collect() 与 toPandas() 的"账落在哪一侧"量给读者看。
# 为什么不 assert：堆增量取决于 GC 什么时候跑、RSS 增量取决于分配器还不还给内核 ——
# 它们**每次跑不会给同一个数**，押它就是给自己造一条与结论无关的红（Ruling(B1) 那一类）。
# 但这一格必须**当场跑**：正文那句"toPandas 的账不只记在 JVM 堆"说的是实测，
# 实测就得有一格打出来（CITE 规矩）。
#
# ⚠ 这一格第一次跑出来的读数是**负的**：`collect()` 之后"堆增量 = -1,820,560 B"。
#   那不是 bug，是**仪器用错了**：`totalMemory() - freeMemory()` 读的是"已分配减去空闲"，
#   而 JVM 不会一有空就 GC —— 上一格（④a）留下的 200 MB 垃圾一直挂在读数上，
#   随后 collect 触发了一次真正的 GC，把那些垃圾一起收走 ⇒ "增量"里混进了**别人**的释放。
#   所以下面每次取基线之前都先 settle（Python 的 gc + JVM 的 System.gc()）。
#   这一条本身就是本节要的课：**峰值与增量都要问"什么时候读的"**，不然会读到负数还以为是省了内存。
N_SMALL = 300_000
small = spark.range(N_SMALL, numPartitions=4).toDF("id")
counted = int(small.agg(f.count(f.lit(1))).take(1)[0][0])  # 用一下 functions as f：确认这份数据真这么多行


def settled_heap_bytes():
    '''取"堆的当前已用"，但先把两侧的垃圾收干净（Python 的引用 + JVM 的堆）。

    非 settle 不可：`totalMemory() - freeMemory()` 里有**还没被 GC 的垃圾**，
    拿它当基线，量到的"增量"会把别人那一笔释放算进来（这一格就这么撞到过负数）。
    '''
    gc.collect()
    spark._jvm.java.lang.System.gc()
    return heap_bytes()


print("对照数据行数", counted, "| 分区数", small.rdd.getNumPartitions())
raw0 = heap_bytes()
h0 = settled_heap_bytes()
r0 = proc_kb("VmRSS")
print("取基线：settle 之前'已用堆'读数 =", raw0, "B；settle 之后 =", h0, "B；差 =", raw0 - h0, "B")
print("   上面那个差就是**还挂在已用堆上的垃圾**。不 settle 就取基线，它会被算进'别人的增量'里 ——")
print("   这一格最初版就是这么量到 -1,820,560 B 这种'collect 反而省了 1.8 MB 堆'的读数的（历史读数，不是本轮的）。")
t1 = time.time()
collected = small.collect()
heap_d_collect = heap_bytes() - h0
rss_d_collect = proc_kb("VmRSS") - r0
per_row_jvm = heap_d_collect / max(counted, 1)
print()
print("[A] collect()   用时", round(time.time() - t1, 2), "s（只打印）")
print("      堆增量 =", heap_d_collect, "B   RSS 增量 =", rss_d_collect, "kB")
print("      每行摊到 JVM 堆 =", round(per_row_jvm, 1), "B/行（本轮 settle 后的算式，不是常数，别当系数背）")
if heap_d_collect < 0:
    print("      ⚠ 这一行是负的 ⇒ settle 没盖住这次的回收节奏，这个'增量'不能当 bytes/行 用（正文那句方法论照样成立）")
del collected
gc.collect()

h1, r1 = settled_heap_bytes(), proc_kb("VmRSS")
t2 = time.time()
pdf_small = small.toPandas()
heap_d_pandas = heap_bytes() - h1
rss_d_pandas = proc_kb("VmRSS") - r1
deep_small = int(pdf_small.memory_usage(deep=True).sum())
print()
print("[B] toPandas()  用时", round(time.time() - t2, 2), "s（只打印）")
print("      堆增量 =", heap_d_pandas, "B   RSS 增量 =", rss_d_pandas, "kB")
print("      pandas deep 用量 =", deep_small, "B（= N * 8 + index 128，与 ④a 同形）")
del pdf_small
gc.collect()
h2 = settled_heap_bytes()
print()
print("[C] settle 之后再读一次堆：", h0, "(A 的基线) ->", h1, "(B 的基线) ->", h2, "(B 释放之后) B")
print("      释放 + settle 把堆读数拉回到 **B 自己的基线**附近（差 =", h2 - h1, "B），但没有回到 h0 ——")
print("      '用完之后回到原点'在 JVM 堆上也不成立：还剩一笔说不清的账留在堆里（本篇不替它编解释，")
print("      只把它打出来 —— 这正是'只盯一个读数就下结论'为什么危险）。而 ④a 那个 VmHWM 是 OS 层的峰值，")
print("      它**根本不跟着掉**，两把尺各管各的。")
print()
print("=> 本格真正稳定的读数只有一句：**堆增量与 RSS 增量不成比例**（A 与 B 各看自己那一行的两个数）。")
print("   两者不同源：堆那本账由 JVM 的 GC 时机决定（settle 之后仍然是估算），")
print("   RSS 那本账还包括 Python 侧的 numpy 列与转换途中的副本 ——")
print("   ⇒ 只看 JVM 堆读数会得出片面的结论（'toPandas 比 collect 省内存'就是那种读法，本轮的数并不支持它）。")
print("   ⚠ 所以这一格一条 assert 都没有：A 与 B 的方向都会随 GC 时机漂，押它就是把判题器交给调度器。")
print("   参照（第一轮探针在**干净 kernel** 上、没有 ④a 那一笔在前时量到的分叉）：")
print("     collect 堆 +75,360,096 B / RSS +55,352 kB，toPandas 堆 +18,268,656 B / RSS +49,504 kB")
print("     —— 那一轮是'堆增量小得多而 RSS 差不多'；本篇不重复押那个方向，只把这一轮的数打在屏幕上。")

## ⑤ 本机边界：为什么本篇不用 `pandas_udf` 量 Python worker

④ 那节说"`toPandas()` 的账落在 Python 进程"，自然的下一步是"那用 `pandas_udf` 把 Python worker
单独拎出来量一份"。**本机不行**，理由是一条边界而不是一条结论：这台镜像的 kernel 用的是
网页 IDE 那份 venv（`sys.executable` 在开局那格打印过），里面装了 pandas 与 numpy，**没有装 pyarrow** ——
而 PySpark 的 pandas API 硬依赖它。于是 `pandas_udf` 这条路在**装饰器**那一步就断了，
连"起一个 Python worker 进程"这个动作都发生不了。

下一格只打印三件事：三个包的读数、以及 `pandas_udf` 这条路当场抛什么。它**不发 marker**：
"这个包没装"不是一条方向性结论，给它发 marker 等于让注册表为一个与判题无关的事实承担双向相等
（纪律照篇 2 的 ⑤a：一件事不许有两个下标）。

由此得到本篇的一条边界声明：**Python worker 的独立进程账、它的 RSS、它和 driver 的 pid 关系 ——
本机未验证**（`local` 模式下 driver 就是这个 kernel 进程，开局那格打的 `pid/ppid` 是唯一量的到的部分）。
到了 K8s 上"python worker 是不是独立容器/独立 limit"那一族问题，同样**未验证**，归 `content/knowledge/`。

In [ ]:
# ⑤ **只打印，不发 marker**（见上一节为什么）。它给 ④ 那句"账落在 Python 侧"补一条前置说明：
# 本机没有第二条路去量 Python worker，所以 ④ 只能用 toPandas 这个"整表落地"的形状。
def try_import(name):
    try:
        m = __import__(name)
        return getattr(m, "__version__", "(没有 __version__)")
    except Exception as e:
        return "MISSING: " + type(e).__name__


print("kernel 用的解释器 =", sys.executable)
for name in ("pandas", "numpy", "pyarrow"):
    print("   ", name, "=", try_import(name))

try:
    from pyspark.sql.functions import pandas_udf

    @pandas_udf("long")
    def plus_one(s):  # noqa: ARG001 这格只为把"能不能构造出来"打出来，函数本身不会被执行
        return s + 1

    print("pandas_udf 构造成功了（与正文不符 => pyarrow 装了，本篇 ⑤ 那节要重写）")
except Exception as e:
    print("pandas_udf 在构造时就抛:", type(e).__name__ + ":", str(e).splitlines()[0])
print()
print("=> 结论（边界，不是判据）：本机量不了 Python worker 那本账。")
print("   这也是本篇 ④ 用 toPandas（整表落地）而不是 pandas_udf（逐批转换）的原因 ——")
print("   前者只需要 pandas + py4j，后者需要这条已经断了的路。")

## ⑥ `container-limit-is-absent-here`：本机这个容器**没有** memory limit
## ⇒ `OOMKilled` 那一族失效模式在这台机器上不存在

这是本篇通向 K8s 那本账的分界线，而且它是一条**关于本机的确定事实**，不是缺陷：

- `/sys/fs/cgroup/memory.max` 的内容是字符串 `"max"`，连读三次都一样
  （`memory.high` 同）。cgroup v2 里 `"max"` 的意思就是**没有设上限** —— 也就是 compose 没给容器
  写 `resources.limits`，K8s 上的对应物是 pod spec 里根本没有 `limits.memory`。
- `/sys/fs/cgroup/memory.events` 的计数器**全是 0**：`low / high / max / oom / oom_kill / oom_group_kill`。
  也就是说这台机器上连"逼近过上限"这件事都没有发生过。

于是本篇能观测到的"内存事故"只剩一种形状：**JVM 抛的 `OutOfMemoryError`**（一个 Java 异常，
能被 catch、能留栈、进程还活着）。而标题那句坏结果里的 `OOMKilled` 是**另一族**东西：
内核/容器运行时把进程 SIGKILL 掉，没有栈、没有异常、退出码 137，K8s 里表现为
pod `Last State: OOMKilled`。**那一族在本机不可能发生**（没有上限就没有触发条件），
所以任何"我在 notebook 里复现了 OOMKilled"的说法在这台机器上都得当场证伪 ——
本篇不试，因为试它就得先给容器加限制，而那是改 compose（不在本篇范围）。

CPU 那本账同形，但**只打印不 assert**：`availableProcessors()` 读到 16（宿主机的核数），
而 `sc.defaultParallelism` 是 2（kernel.json 里 `local[2]` 写的）。⚠ 这两个数不相等是**设计如此**，
押它等于把结论写成"这台机器配错了"；它要教的只是"看得见核数 ≠ 管得了限流"——
K8s 上 `cpu limit` 靠 CFS 配额节流，`availableProcessors` 到那时读的是**宿主**还是**配额**，
本机关起门来无从验证（未验证）。

⚠ 判据的严格性：简报给的形状是 `len(set(三次读数)) == 1`，但那一条**三次都读到异常字符串也会点头**。
所以这里判的是 `set(三次) == {"max"}`（既稳定、又确实是 `"max"`），并在红话里说清
"红 = compose 加了上限 = 本节正文过期"，不是"判据太严"。

In [ ]:
# marker container-limit-is-absent-here gate 的是**一句**结论：本机这个容器没有 memory limit，
# 所以 OOMKilled 那一族失效模式在这台机器上不存在（这是通向 K8s 那本账的分界，不是缺陷）。
# assert 只押两件事（简报点名的那两个确定量）：memory.max 稳定且确实是 "max"、
# memory.events 的计数器一个都没动。CPU 那一对只打印（理由见上一节：不等是设计如此）。
CG = "/sys/fs/cgroup"
reads = []
for _ in range(3):
    with open(os.path.join(CG, "memory.max"), "r", encoding="utf-8") as fh:
        reads.append(fh.read().strip())
events = {}
with open(os.path.join(CG, "memory.events"), "r", encoding="utf-8") as fh:
    for ln in fh.read().splitlines():
        k, sep, v = ln.partition(" ")
        if sep:
            events[k] = int(v.strip())

with open(os.path.join(CG, "memory.high"), "r", encoding="utf-8") as fh:
    high_now = fh.read().strip()
with open(os.path.join(CG, "memory.current"), "r", encoding="utf-8") as fh:
    current_now = fh.read().strip()
print("cgroup v2 路径       =", CG, "（" + ", ".join(os.path.join(CG, n) for n in ("memory.max", "memory.events")) + " 都在）")
print("memory.max 连读三次   =", reads, "| 集合 =", sorted(set(reads)))
print("memory.high          =", repr(high_now), "(只打印：与 max 同形，本机也没设软上限)")
print("memory.current       =", current_now, "B (只打印：**整个容器**的当前用量，不是本篇进程的)")
print("memory.events        =", events)
print()
print("v1 那一族路径在这台机器上不存在（只打印，它判的是这台机器的文件系统布局，不是本篇的结论）：")
for name in ("memory/memory.limit_in_bytes", "memory/memory.usage_in_bytes", "memory/memory.failcnt"):
    try:
        with open(os.path.join(CG, name), "r", encoding="utf-8") as fh:
            print("   ", name, "=", fh.read().strip())
    except Exception as e:
        print("   ", name, "->", type(e).__name__, "（这台是 v2）")
print()
print("CPU 那本账（**只打印，一条都不 assert**）：availableProcessors =", rt.availableProcessors(),
      "| defaultParallelism =", sc.defaultParallelism)
print("   两个数不等是设计如此（核数来自宿主，并行度来自 kernel.json 的 local[2]）；")
print("   它要说的只是'看得见核数 != 管得了限流'，K8s 上 cpu limit 读什么：本机无从验证（未验证）。")
print()
print("=> 本机可观测的内存事故只有 JVM 的 OutOfMemoryError（能被 catch、有栈、进程活着）；")
print("   OOMKilled（SIGKILL、无栈、退出码 137）在这一族计数器全 0、上限是 max 的机器上**没有触发条件**。")

assert len(reads) == 3 and set(reads) == {"max"}, (
    f"memory.max 三次读数是 {reads}（集合 {sorted(set(reads))}）⇒ 两种坏法之一："
    "①compose 给容器加了 resources.limits（那本节正文与导语那条'本机看不见 OOMKilled'当场过期，"
    "要按新值重写并把'撞上限'这一族收进判据）；②这台机器的 cgroup 不是 v2 或读不到那个路径"
    "（那是环境问题，不是结论坏了 —— 红话里的读数会告诉你读到了什么）。别把这条改成'read 到的值'"
)
assert events.get("oom_kill") == 0, (
    f"memory.events 里 oom_kill = {events.get('oom_kill')} ⇒ 这台容器**发生过** OOM kill，"
    "本节标题那句'那一族失效模式在这台机器上不存在'当场过期（正文要整段重写：那时本机就能量了）"
)
assert events.get("oom") == 0, f"oom 计数器 = {events.get('oom')}，不是 0 ⇒ 同上：触发条件出现过，本节的分界线要重画"
print("WI90[04-memory-accounts-and-the-boot-time-ceiling][container-limit-is-absent-here] OK", flush=True)

## 速查表（可以只靠它回答问题）

| 那本账 | 本机存在吗 | 怎么读（本镜像实测） | 坏法 / 注意点 |
| --- | --- | --- | --- |
| driver 堆上限 | 存在，**只在启动期** | `Runtime.getRuntime().maxMemory()` = 536,870,912 B（① 当场打印） | 运行时 `set('spark.driver.memory')` 抛 `CANNOT_MODIFY_CONFIG`，且抛完上限一字不变 |
| 配置项的生效值 | 存在 | 单参 `spark.conf.get(k)` = 生效值（含注册项缺省） | **双参 `get(k, default)` 会给 default 做类型校验，且它顶替注册项缺省**：`get(k)` 200 / `get(k,"999")` "999" / `get(k,"(unset)")` 抛 |
| executor 内存 / 核数 | **不存在** | `spark.conf.get("spark.executor.memory")` 抛 `SQL_CONF_NOT_FOUND`；`sc.getConf().get(..., d)` 回 d | 别把"读不到"读成"没配好"：local 模式没有这本账 |
| 并行度 | 存在（作为活值） | `sc.defaultParallelism` = 2 | 有活值 ≠ 有配置项：`spark.default.parallelism` 在 SparkConf 里查无此键 |
| Python 侧的峰值 | 存在 | `/proc/self/status` 的 `VmHWM`（峰值）与 `VmRSS`（当前）；`ru_maxrss` 与 `VmHWM` 同一个数 | ④ 实测：2,000,000 行单列 int64 的**净** 16 MB，峰值增量是它的 35~36 倍；`del + gc` 后 HWM 永不掉，而 RSS 只掉 MB 级、**方向都不保证**（六次里一次不降反升）⇒ 押"同一瞬间 HWM−RSS 差一个数量级"，别押降幅 |
| container 上限 | **本机没设** | `/sys/fs/cgroup/memory.max` = `"max"`；`memory.events` 全 0 | 所以 `OOMKilled` 这一族在本机不可观测；能撞的只有 JVM 的 `OutOfMemoryError` |
| Python worker 独立进程 | **量不了** | 没有 pyarrow ⇒ `pandas_udf` 在构造时就抛（⑤ 当场打印） | 这一本账与 K8s 上"worker 是否独立容器"都未验证 |

### 追问链（面试往哪走，以及哪一步本机答不了）

1. "那你本地怎么防止 `toPandas()` 把 driver 打爆？" —— 先说**判据**再说解法：净大小是
   `行数 × 列宽 + index`（④a 那条可纯算的等式），峰值是它的几十倍，所以"200 万行才 16 MB"
   这句话在这里是**误导性**的；解法是加 `limit()` / 走 `write.parquet` / 用 `take(n)` 而不是 `collect()`。
   ⚠ 具体阈值（多少行必须离开 toPandas）本篇**不给数**：它取决于列数、类型、以及这一版有没有 pyarrow，
   那是要按作业量的东西，不是教程该背的常数。
2. "`spark.driver.memory` 改不了，那生产上到底改哪里？" —— 改**启动命令**（`spark-submit --driver-memory`
   / K8s 的 driver pod `resources.limits.memory`）。⇒ 从这里开始本篇**答不了**：本机没有 pod，
   "limit 与 `-Xmx` 该给多大余量（JVM 之外还有 metaspace/线程栈/direct buffer/Python）"这一族是
   `content/knowledge/` 那一半的内容，本篇只证明了一件事 —— 运行时那个旋钮是焊死的。
3. "怎么证明你释放了内存？" —— 看 `VmRSS` 的**量级**与 `VmHWM - VmRSS` 那个差，别用 `VmHWM`/`ru_maxrss`
   当"当前用量"（它们是峰值），也别押"`del` 之后 RSS 一定降"——本篇六次实测里有一次是**涨**的（④a 打印那行数）。
4. "K8s 上 OOMKilled 和 OutOfMemoryError 你怎么分辨？" —— 本机只能给出**一半**答案：
   异常这一族有栈、进程活着；SIGKILL 这一族在 `memory.events` 全 0、上限是 `max` 的机器上
   没有触发条件（⑥ 判的就是这个"没有"）。真分辨要靠 pod 的 `lastState`，**本机未验证**。

### 三条本篇**故意没有**做的事（写了就是假话）

- 没有"撞一次堆上限看看会怎样"的格子：local 模式 driver 就是这个 kernel，炸了后面全废（纪律）。
- 没有 assert 任何墙上时间：④a/④b 的秒数只打印（Ruling(B1)）。
- 没有给 pyarrow 缺失发 marker：那是边界不是结论（⑤ 那节写了为什么）。

## 题库指针（同一套口径，不是"再找篇资料"）

- **`bd-pyspark-0002`** ——「join 右表多版本导致行数膨胀：先收敛维度再广播（结果行数必须等于左表）」
  （senior，判题 `pyspark`）。它是本篇 ④ 那本账的**反面教材**：题面明令
  "禁止 `collect()` / `toPandas()`"，而本篇正好量到了为什么 —— 峰值是净结果的几十倍（④a），
  而且那本账不全在 JVM 堆里（④b）。同时它是篇 1 的 ③B / 篇 3 的 ③ 那道"广播的资格"题：
  维度表带多版本时行数先膨胀，`sizeInBytes` 越过阈值就不自动广播，硬广播就是 driver/executor OOM。
  ⇒ 三篇各量一头，口径是同一句：**先收敛维度，再谈广播/落地**。
- **`bd-rubric-0008`** ——「成本治理：把『每月 $1.2M 的数据账单』变成可归因、可优化、可防回弹的系统」
  （判题 `llm-rubric`）。它第 3 条要的正是"省 A 涨 B"的实例，其中两条就是本篇的账：
  **"盲目 cache() → 内存挤压与静默重算"**（④a 那格把"净大小"与"峰值增量"分成两个数量看 ——
  那正是记 `cache()` 这本账唯一不会骗人的量法：一份 RDD 落地后**占**的字节，与为了让它落地而**走过**的
  副本，是两个数，而只打印前者的人会把后者当成 0）与**"广播大表 → driver/executor OOM"**。
  ⚠ 它要求的是**归因口径**（spill / GC / per-task 分布），本篇只给出"本机能量到的那几本账"，
  真作业的那些指标（shuffle bytes、spill 次数）本机**没有**（`local` 模式不落 shuffle 文件到网络，
  也没有节点级指标）—— 那一条去题里答，别拿本篇的绿当它的证据。

系列内分工：篇 1 = 倾斜与热点 key；篇 2 = 小文件与物理设计；篇 3 = 计划解读与 AQE 三段能力；
**本篇 = 内存与配置的账**（哪几本存在、哪几本在这台机器上根本不存在）。
本篇刻意**只**cache 一次（④a 那份 2,000,000 行的数据），且用完立刻 `unpersist()`：
多 cache 一份，④a 量到的峰值里就多一笔说不清来源的账 —— 那正是本节要拆穿的东西，不能自己制造。

In [ ]:
# 最后一个单元必须是它：闸门（tutorials.test.ts）跑完这一篇就结束，
# 不往容器里丢活着的 Spark 内核（常驻那组那条结构性的判据判的就是它，宿主档就能红）。
spark.stop()
print("stopped")